# Correlated-state calibration of the residual certificate

Operational calibration of the deterministic bias certificate from the connected-layer-sector v02 manuscript on a correlated 1D Hubbard ground state, using exact diagonalization in the noninteracting single-particle eigenbasis as the dictionary basis.

**Scope (read first):** this notebook is

- **not** a theorem validation (the theorems in the manuscript are algebraic and require no numerical validation);
- **not** a complete chemistry-workflow benchmark;
- **not** evidence that $\Delta_{r,U(1)}$ is small for general correlated chemistry states;
- **not** a measurement-advantage claim (uses exact diagonalization, no measurement records).

It is an exact-numerical scale check on a tunable correlated fermionic model, extending the implementation audit (Sec. VI of the manuscript) from abstract fixed-$N$ test states to one correlation-controlled lattice model.

Runs in roughly four minutes on a laptop CPU.

## Setup

**Model.** Open-boundary 1D Hubbard chain with $n_\text{sites} = 4$, half-filled ($N_\uparrow = N_\downarrow = 2$ in $8$ spin-orbitals, 256-dimensional full Hilbert space, 36-dimensional $(N_\uparrow, N_\downarrow) = (2, 2)$ sector):

$$H = -t \sum_{i,\sigma} (a_{i,\sigma}^\dagger a_{i+1,\sigma} + \text{h.c.}) + U \sum_i n_{i,\uparrow} n_{i,\downarrow}$$

with $t = 1$ and $U/t \in \{0, 1, 4, 8, 16\}$.

**Dictionary basis.** The catalog is evaluated in the $U = 0$ noninteracting single-particle eigenbasis of the same tight-binding chain, fixed across the $U/t$ sweep. At $U = 0$ the half-filled ground state in this basis is the lowest-two-modes-per-spin Slater determinant, which is occupation-basis diagonal. By Section V of the manuscript, $\Delta^\text{cat}_{4,U(1)}$ vanishes exactly there. At $U/t > 0$ the ground state in the same basis is correlated.

**State preparation.** For each $U/t$ value the site-basis Hamiltonian is JW-encoded and projected to the $(2,2)$ sector; the ground state is obtained by `numpy.linalg.eigh`. The state vector is then rotated into the $U = 0$ single-particle eigenbasis using the many-body unitary $V$ that implements the one-body orbital rotation $a_p^\dagger \mapsto \sum_q W_{qp} c_q^\dagger$, constructed numerically as $\exp(\sum_{pq} (\log W)_{pq} a_p^\dagger a_q)$ on the 8-qubit Hilbert space.

In [ ]:
import json
import sys
from pathlib import Path

# Locate the project root from this notebook (notebooks/<this>.ipynb)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "calibration_chemistry.json"
CODE = ROOT / "code"

print(f"project root: {ROOT}")
print(f"calibration JSON: {DATA} ({'exists' if DATA.exists() else 'missing'})")

## Load (or regenerate) the calibration data

The next cell loads the precomputed JSON. If you want to regenerate it from scratch, set `REGEN = True` — that calls `code/calibration_chemistry.py` with the standard sweep and writes a fresh JSON. The default flow is to load the deposited result and verify it.

In [ ]:
REGEN = False  # set True to regenerate from scratch (~4 min on CPU)

if REGEN:
    sys.path.insert(0, str(CODE))
    from calibration_chemistry import hubbard_calibration
    out = {
        "schema_version": 1,
        "implementation_status": {
            "hubbard": "implemented",
            "h2_sto3g": "deferred (PySCF unavailable)",
            "lih_sto3g": "deferred (PySCF unavailable)",
        },
    }
    out.update(hubbard_calibration(n_sites=4,
                                   Ut_values=(0.0, 1.0, 4.0, 8.0, 16.0)))
    DATA.write_text(json.dumps(out, indent=2), encoding="utf-8")

data = json.loads(DATA.read_text(encoding="utf-8"))
print("schema_version:", data.get("schema_version"))
print("generated:", data.get("generated"))
print("implementation_status:", json.dumps(data["implementation_status"], indent=2))

## Calibration table

Metrics, all evaluated on $\rho = V|\psi_\text{GS}\rangle\langle\psi_\text{GS}|V^\dagger$ in the $U = 0$ eigenbasis:

- $\Delta^\text{cat}_{4,U(1)}(\rho) = \sup_W |\kappa_{[|W|]}(W;\rho)|$ over the chemistry catalog of Definition 3
- $\max_W |\tau^G_W(\rho)|$ — the worst-case residual on the catalog
- $B^\text{eff}_\text{max} = \max_W |\tau^G_W| / \Delta^\text{cat}$ — empirical effective constant
- $\eta_\text{universal} = \max_W |\tau^G_W| / (B_4 \cdot \Delta^\text{cat})$ with $B_4 = 105$ — certified universal-bound ratio

In [ ]:
rows = data["hubbard"]
header = ("U/t", "E_0", "Delta^cat", "max|tau|", "B_eff_max", "eta_universal")
print("  ".join(f"{h:>14}" for h in header))
print("  ".join(["-" * 14] * len(header)))
for r in rows:
    eta = r["eta_universal"]
    eta_s = f"{eta:14.4e}" if eta == eta else "   n/a (zero)"
    beff = r["B_eff_max"]
    print(f"{r['U_over_t']:>14.2f}  "
          f"{r['E_ground']:>14.6f}  "
          f"{r['delta_cat']:>14.4e}  "
          f"{r['max_tau']:>14.4e}  "
          f"{beff:>14.4f}  "
          f"{eta_s}")

## Reading the table

**Sec V baseline reproduced.** The $U = 0$ row gives $\Delta^\text{cat} \approx 3 \times 10^{-16}$ — floating-point zero. The Hubbard $U=0$ ground state in the chosen dictionary basis IS a single Slater determinant occupation-diagonal in that basis, so by Section V the catalog envelope must vanish identically. The numerical zero confirms the basis-rotation pipeline is correct end-to-end and the Section V theorem is recovered exactly on this state.

**Correlation dependence.** $\Delta^\text{cat}$ grows monotonically with $U/t$ from $\sim 0.05$ at weak correlation to $\sim 0.31$ at $U/t = 16$, saturating in the Mott-insulator limit. This is the operational signal the certificate is designed to produce: a calibrated, dictionary-basis-dependent measure of how much higher-cumulant content the chosen order-$\le 2$ closure leaves uncontrolled.

**Universal-bound looseness.** The certified ratio $\eta_\text{universal} \approx 0.0095$ at every $U/t > 0$ point, so the universal partition-lattice constant $B_4 = 105$ is about two orders of magnitude looser than tight on this catalog and this state. This is consistent with the implementation audit (Sec. VI of the manuscript), which reports $B^\text{eff}_\text{max} \approx 2.0$ on the abstract fixed-$N$ suite, again indicating that the per-word block-refined constants $\widehat B^\text{charge}_4(W) \in \{1, 3, 5\}$ from Corollary 1 are the operationally tighter bound, not the universal $B_4$.

**Tight-residual structure.** On every row at $U/t > 0$, $\max_W |\tau^G_W|$ equals $\Delta^\text{cat}$ to floating-point precision. The worst-case residual is dominated by the top-partition cumulant of a single catalog word, with non-distinguished-block factors negligible relative to the distinguished neutral high cumulant — exactly the structure the block-refined bound (Theorem 3) is designed to exploit.

## Sanity checks built into this notebook

Three exact-numerical assertions a reader can verify without re-running the calibration:

In [ ]:
# 1. U=0 row reproduces Sec V zero baseline to floating-point precision.
u0 = next(r for r in rows if r["U_over_t"] == 0.0)
assert u0["delta_cat"] < 1e-12, f"U=0 baseline failed: {u0['delta_cat']}"
print(f"PASS: U=0 baseline Delta^cat = {u0['delta_cat']:.2e} < 1e-12")

# 2. Delta^cat is monotonically nondecreasing in U/t.
deltas = [r["delta_cat"] for r in rows]
assert all(deltas[i+1] >= deltas[i] - 1e-12 for i in range(len(deltas)-1)), \
    "Delta^cat is not monotone in U/t"
print(f"PASS: Delta^cat monotone nondecreasing in U/t: {[f'{d:.4f}' for d in deltas]}")

# 3. eta_universal stays well below 1, confirming the universal bound is loose.
etas = [r["eta_universal"] for r in rows if r["eta_universal"] == r["eta_universal"]]
assert max(etas) < 0.1, f"eta_universal exceeded 0.1: {max(etas)}"
print(f"PASS: eta_universal max = {max(etas):.4f} < 0.1 (universal B_4=105 is loose by ~100x)")

## Deferred cells

The original calibration plan included Cell 2 (H$_2$/STO-3G FCI swept across bond length) and Cell 3 (LiH/STO-3G FCI at equilibrium). Both require chemistry one- and two-electron integrals from a Hartree–Fock calculation in the canonical RHF molecular-orbital basis, which in turn requires PySCF. The local environment used to produce this notebook could not build PySCF from source under the available Python distribution; OpenFermion is installed but its molecular-data path also relies on PySCF.

These two cells are deferred. In a follow-up environment with PySCF available, the same evaluation pipeline applies unchanged: for each geometry, build the FCI ground state in the canonical RHF MO basis (so the Hartree–Fock determinant gives $\Delta^\text{cat} = 0$ exactly by Section V), evaluate the same metrics. Equilibrium points are expected to give small but nonzero $\Delta^\text{cat}$ reflecting weak multi-reference correction; stretched bond lengths and the LiH equilibrium are expected to give larger values.

## Reproducibility

- This notebook lives at `notebooks/calibration_chemistry.ipynb`.
- The calibration code lives at `code/calibration_chemistry.py` and reuses helpers from `code/screen_sector_cumulant_theorem.py` (JW operators, fixed-$N$ projector, partition-lattice Möbius inversion, catalog enumeration).
- The deposited result lives at `data/calibration_chemistry.json` with SHA256 checksum recorded in `MANIFEST.json`.
- Determinism: ED with explicit Hamiltonian construction; no random seeds.
- Re-run from scratch: set `REGEN = True` in the load cell above, or run `python code/calibration_chemistry.py --n-sites 4 --Ut 0 1 4 8 16` directly.